# OnT quickstart: ontology embeddings in four steps

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HuiYang1997/OnT/blob/main/notebooks/OnT_Quickstart.ipynb)

**OnT** (Ontology Transformer, ISWC 2025) fine-tunes a sentence encoder so that concepts become points in a hyperbolic (Poincaré) ball:
general concepts sit near the centre, specific ones near the boundary, and $C \sqsubseteq D$ becomes "$C$ close to $D$, and further out".

1. **Install** the `ontology-transformer` package
2. **One click**: load a small family ontology and train OnT on it
3. **Embed** concepts, including $\exists r.C$
4. **Score** subsumptions $C \sqsubseteq D$ and rank candidate parents

Runs on the free Colab **CPU** runtime (a GPU is optional: *Runtime → Change runtime type*). Choose **Runtime → Run all**.

## Step 1 · Install

`ontology-transformer` reads OWL files through the OWL API, so it needs Java.

In [1]:
import os, sys, subprocess

if "google.colab" in sys.modules:                       # Java is not preinstalled on Colab
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null",
                   shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

%pip install -q ontology-transformer==0.1.8

import logging, warnings           # keep the output short; delete these two lines to see all training logs
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)

Note: you may need to restart the kernel to use updated packages.


## Step 2 · One click: load an ontology and train OnT

`family.ofn` is the toy family ontology of TransBox (WWW 2025), the running example of the tutorial:
6 classes, 2 roles and 9 axioms ($\alpha_1$–$\alpha_9$) in OWL functional syntax. Every class has a readable label, because OnT encodes **labels**.

In [2]:
import urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/HuiYang1997/OnT/main/notebooks/family.ofn", "family.ofn")

for line in open("family.ofn"):                     # the nine axioms
    if line.startswith(("SubClassOf", "DisjointClasses")):
        print(line, end="")

SubClassOf(:Father ObjectIntersectionOf(:Male :Parent))
SubClassOf(:Mother ObjectIntersectionOf(:Female :Parent))
SubClassOf(ObjectIntersectionOf(:Male :Parent) :Father)
SubClassOf(ObjectIntersectionOf(:Female :Parent) :Mother)
DisjointClasses(:Male :Female)
DisjointClasses(:Parent :Child)
SubClassOf(:Child ObjectSomeValuesFrom(:hasParent :Mother))
SubClassOf(:Child ObjectSomeValuesFrom(:hasParent :Father))
SubClassOf(:Parent ObjectSomeValuesFrom(:hasChild :Child))


Now the one line that matters: `fit` converts the OWL axioms into training data, fine-tunes the encoder
(default base model: `all-MiniLM-L12-v2`, 384 dimensions) and picks the best centripetal weight $\lambda$ for scoring.
OnT has no loss for disjointness, so the two `DisjointClasses` axioms ($\alpha_5$, $\alpha_6$) are not used.

In [3]:
from ont import OntologyTransformer

model = OntologyTransformer.fit(owl_path="family.ofn", output_dir="ont_family",
                                num_epochs=100, batch_size=8, learning_rate=5e-4,
                                eval_ratio=0.0)      # tiny ontology: train on every axiom

print("best lambda:", model.best_lambda)

Download `en_core_web_sm` for pos tagger.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/12.8 MB ? eta -:--:--

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 55.3 MB/s  0:00:00


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Epoch,Training Loss,Validation Loss,Axiom Kind,Centri Weight,H@1,H@10,H@100,Mrr,Mr,Median,Auc
1,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.900000
2,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.900000
3,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.900000
4,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.900000
5,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.900000
6,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.900000
7,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.900000
8,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.900000
9,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.900000
10,No log,No log,nf1,0.000000,0.000000,1.000000,1.000000,0.500000,2.000000,2.000000,0.900000


best lambda: 1.5


## Step 3 · Embed concepts

`encode` maps any label to a point in the Poincaré ball. Its hyperbolic distance from the origin acts as a **depth**:
general concepts should be closer to the centre than specific ones.
`encode_existence(r, C)` embeds the complex concept $\exists r.C$ by a learned rotation of $C$.

In [4]:
import pandas as pd
import torch

concepts = ["parent", "male", "female", "father", "mother", "child"]
emb = model.encode(concepts, convert_to_tensor=True)
print("embedding matrix:", tuple(emb.shape))

depth = model.manifold.dist0(emb)              # hyperbolic distance to the origin
pd.DataFrame({"concept": concepts,
              "depth": depth.cpu().numpy().round(2),
              "first 4 coordinates": ["  ".join(f"{x:+.3f}" for x in v[:4]) for v in emb.cpu().numpy()]})

embedding matrix: (6, 384)


,concept,depth,first 4 coordinates
0,parent,8.30,+0.062 +0.408 -0.139 -0.329
1,male,8.21,+0.224 +0.138 -0.182 -0.525
2,female,8.85,-0.073 +0.297 +0.431 +0.147
3,father,8.92,+0.086 +0.432 -0.153 -0.379
4,mother,8.69,-0.040 +0.130 +0.392 +0.284
5,child,13.43,+0.259 +0.306 -0.584 -0.799


In [5]:
has_male_parent = model.encode_existence("has parent", "male")      # ∃hasParent.Male
print("∃ has parent . male ->", has_male_parent.shape, "  ".join(f"{x:+.3f}" for x in has_male_parent[0, :4]), "...")

∃ has parent . male -> (1, 384) +0.061  +0.109  +0.126  +0.518 ...


## Step 4 · Score subsumptions

`score_hierarchy(C, D, λ)` returns $-\big(d(C,D) + \lambda\,(\lVert D\rVert - \lVert C\rVert)\big)$: **higher means $C \sqsubseteq D$ is more plausible**.
Compare candidate parents **of the same concept**: some are stated in the ontology, some are false (Father and Mother are disjoint).
Scores are plausibility, not proofs.

In [6]:
lam = model.best_lambda or 0.0

def score(child, parent):
    c = model.encode([child], convert_to_tensor=True)
    p = model.encode([parent], convert_to_tensor=True)
    return round(model.score_hierarchy(c, p, weight=lam).item(), 2)

rows = [("father", "male",   "stated (α1)"),
        ("father", "parent", "stated (α1)"),
        ("father", "mother", "false (α1, α2, α5)"),
        ("father", "child",  "false (α1, α6)"),
        ("mother", "female", "stated (α2)"),
        ("mother", "male",   "false (α2, α5)")]
pd.DataFrame([(f"{c} ⊑ {p}", status, score(c, p)) for c, p, status in rows],
             columns=["axiom", "status", "score"])

,axiom,status,score
0,father ⊑ male,stated (α1),-8.57
1,father ⊑ parent,stated (α1),-0.07
2,father ⊑ mother,"false (α1, α2, α5)",-11.20
3,father ⊑ child,"false (α1, α6)",-18.74
4,mother ⊑ female,stated (α2),-3.98
5,mother ⊑ male,"false (α2, α5)",-11.83


The same score ranks **all** classes as candidate parents, for an existing class or for a concept that is not in the ontology yet:

In [7]:
import json
all_labels = list(json.load(open("ont_family/data/concept_names.json")).values())   # class labels, written by fit()
cand = model.encode(all_labels, convert_to_tensor=True)

def top_parents(query, k=3):
    q = model.encode([query], convert_to_tensor=True)
    s = model.score_hierarchy(q.expand_as(cand), cand, weight=lam)
    return [all_labels[i] for i in s.argsort(descending=True).tolist() if all_labels[i] != query][:k]

for query in ["father", "mother", "grandfather", "son"]:      # "grandfather" and "son" are not in the ontology
    print(f"{query:>12} ⊑ ?  ", top_parents(query))

      father ⊑ ?   ['parent', 'male', 'female']
      mother ⊑ ?   ['female', 'parent', 'male']
 grandfather ⊑ ?   ['parent', 'father', 'male']
         son ⊑ ?   ['male', 'parent', 'father']


In [8]:
# The running query q: Child ⊑ ∃hasParent.Male, entailed by α1 and α8 but never stated
target = torch.tensor(has_male_parent, device=cand.device)
for c in ["child", "parent"]:
    s = model.score_hierarchy(model.encode([c], convert_to_tensor=True), target, weight=lam).item()
    print(f"{c:>6} ⊑ ∃ has parent . male : {s:6.2f}")

 child ⊑ ∃ has parent . male :  -7.54
parent ⊑ ∃ has parent . male : -11.69


### Notes

* Scores are **plausibility**, not proofs: check any new axiom with an OWL reasoner (e.g. ELK, HermiT).
* **Your own ontology**: replace `family.ofn` by any OWL file (`.owl`, `.ofn`, `.ttl`), e.g. uploaded via the Colab file panel.
* **No training**: load a released model, e.g. `OntologyTransformer.from_pretrained("Hui97/OnT-MiniLM-L12-galen")` (also `-go`, `-anatomy`, and `OnT-MPNet-*`).
* Paper: Yang, Chen, He, Gao & Horrocks. *Language Models as Ontology Encoders*. ISWC 2025. Code: github.com/HuiYang1997/OnT